# 1. Entrenamiento y registro de experimentos

Este notebook guía el flujo principal del laboratorio de análisis de sentimientos sobre **IMDB Movie Reviews en español**.

Aquí no vamos a reescribir toda la lógica del proyecto dentro de celdas sueltas. En cambio, vamos a usar el paquete del repositorio para mantener un flujo reproducible, limpio y fácil de defender.

## Objetivos de este notebook

- Entender la configuración global del proyecto.
- Ver cómo se prepara el dataset desde Kaggle.
- Ejecutar el baseline clásico.
- Ejecutar los modelos neuronales con `TextVectorization`.
- Registrar cada corrida en MLflow.

## Nota importante sobre la consigna

La guía menciona 7 experimentos, pero la suma explícita de requisitos da 8 corridas:

- 1 baseline
- 1 modelo con embeddings entrenables
- 3 embeddings preentrenados × 2 variantes (congelado y ajustado)

Por eso la configuración del proyecto deja preparadas **8 corridas**.

## Preparación del entorno

Antes de ejecutar el notebook, asegúrate de haber creado el entorno virtual y de haber instalado las dependencias:

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
python -m spacy download es_core_news_md
python -m spacy download es_core_news_lg
```

Si vas a registrar corridas en una máquina dedicada, también debes definir `MLFLOW_TRACKING_URI`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print(PROJECT_ROOT)

## Revisar la configuración global

Toda la parametrización del laboratorio está centralizada en `configs/experiments.yaml`.

Eso nos ayuda a justificar con claridad:

- la longitud máxima de secuencia,
- el tamaño del embedding,
- el número de capas ocultas y neuronas,
- las fuentes de embeddings preentrenados,
- y los nombres de las corridas que se verán luego en MLflow.

In [ ]:
from imdb_sentiment.config import load_config

config = load_config(PROJECT_ROOT / "configs/experiments.yaml")
print("Experimento de MLflow:", config.tracking.experiment_name)
print("Longitud máxima de secuencia:", config.neural.sequence_length)
print("Tamaño del embedding entrenable:", config.neural.embedding_dim)
print("Capas ocultas:", config.neural.hidden_units)
print("Experimentos definidos:")
for experiment in config.experiments:
    print("-", experiment.name, "->", experiment.description)

## Carga y partición del dataset

El dataset se descarga usando `kagglehub`, tal como solicita la consigna. Después se limpian filas vacías, se localiza automáticamente la columna de texto y la columna de sentimiento, y finalmente se hace una partición estratificada:

- 80% entrenamiento
- 10% validación
- 10% prueba

La estratificación es importante porque garantiza que la proporción de clases se conserve en los tres conjuntos.

In [ ]:
from imdb_sentiment.data import prepare_dataset_splits

splits = prepare_dataset_splits(config.dataset, config.splits)
print("Columna de texto:", splits.text_column)
print("Columna de etiqueta:", splits.label_column)
print("Tamaño train:", len(splits.x_train))
print("Tamaño validación:", len(splits.x_val))
print("Tamaño test:", len(splits.x_test))
print("Distribución de clases:")
print(splits.dataset_metadata["class_distribution"])

## Baseline: TF-IDF + Regresión Logística

Este baseline suele ser fuerte en análisis de sentimientos porque explota directamente palabras y n-gramas muy asociados a polaridad, por ejemplo términos como *excelente*, *aburrida*, *recomendable* o combinaciones frecuentes de dos palabras.

La idea del baseline no es solo tener un punto de partida, sino ofrecer una referencia sólida contra la cual comparar el beneficio real de usar embeddings y redes neuronales.

In [ ]:
!python ../scripts/train_all.py --config ../configs/experiments.yaml --only baseline_tfidf_logreg

## Modelos neuronales con `TextVectorization`

Todos los experimentos neuronales usan `keras.layers.TextVectorization`, cumpliendo directamente la restricción de la consigna.

La arquitectura base es la misma para todos:

- entrada de texto crudo,
- vectorización a secuencias de enteros,
- capa de embeddings,
- `GlobalAveragePooling1D`,
- dos capas densas ocultas,
- salida sigmoide para clasificación binaria.

En los modelos preentrenados sólo cambia la capa de embeddings.

In [ ]:
!python ../scripts/train_all.py --config ../configs/experiments.yaml

## Qué revisar en MLflow después de entrenar

Cada corrida debe guardar al menos:

- parámetros relevantes,
- métricas de validación y prueba,
- artefactos como matriz de confusión, curva ROC e historial de entrenamiento,
- el modelo entrenado listo para despliegue.

El siguiente notebook se enfoca precisamente en comparar esas corridas y construir el análisis interpretativo.